# Vollständiger Pipeline-Test

Führt alle Schritte durch und validiert Outputs nach jedem Schritt.

## 1) Data Import
Lade Rohdaten, bereinige Labels und konvertiere numerische Spalten.

In [ ]:
!python ../scripts/dataload.py

In [ ]:
from FIT_python.data_import_wrapper import DataImporter
from FIT_python.config import RAW_DIR, DEFAULT_TARGETS
import pandas as pd

# Initialisiere Importer
importer = DataImporter(
    raw_dir=RAW_DIR,
    target_cols=DEFAULT_TARGETS,
    skip_fillna=['trail'],
    label_map={'f':'female','m':'male'}
)
dfs = importer.run()

# Zeige Formen und Kopfzeilen
for name, df in dfs.items():
    print(f"Dataset '{name}': {df.shape}")
    display(df.head())

## 2) Splits erstellen
Erzeuge Train/Test/Inference Splits.

In [ ]:
!python ../scripts/create_splits.py

In [ ]:
from pathlib import Path
from FIT_python.config import SPLITS_DIR, normalize_dataset_name

print("Splits-Verzeichnis:")
for ds in SPLITS_DIR.iterdir():
    if ds.is_dir():
        files = sorted(p.name for p in ds.glob("*.parquet"))
        print(f"  {normalize_dataset_name(ds.name)}: {files}")

## 3) Landmark Mapping
Erzeuge Landmark- und Point-Maps für Otter.

In [ ]:
!python ../scripts/create_landmark_map.py

In [ ]:
from FIT_python.config import OTTER_LANDMARK_MAP_PATH, OTTER_POINT_MAP_PATH

assert OTTER_LANDMARK_MAP_PATH.exists(), 'Landmark map missing'
assert OTTER_POINT_MAP_PATH.exists(), 'Point map missing'
print('Landmark and point maps generated')


## 4) Scaling
Wende Scaler auf alle Splits an.

In [ ]:
!python ../scripts/scale_splits.py

In [ ]:
from FIT_python.config import PROCESSED_DIR, normalize_dataset_name
import pandas as pd

scaled = []
print('Skalierte Splits:')
for ds in PROCESSED_DIR.iterdir():
    train_file = ds / 'train.parquet'
    if train_file.exists():
        df = pd.read_parquet(train_file)
        scaled.append(ds.name)
        print(f'  {normalize_dataset_name(ds.name)}: {df.shape}')
assert scaled, 'No scaled datasets found'


## 5) Transformation → NumPy-Arrays
Konvertiere zu X_*.npy und y_*.npy.

In [ ]:
!python ../scripts/transform_splits.py

In [ ]:
from FIT_python.config import NUMERIC_DIR, normalize_dataset_name

print('Numeric arrays:')
datasets = []
for ds in NUMERIC_DIR.iterdir():
    if ds.is_dir():
        files = sorted(p.name for p in ds.glob('*.npy'))
        if files:
            datasets.append(ds.name)
        print(f'  {normalize_dataset_name(ds.name)}: {files}')
assert datasets, 'No numeric arrays created'


## 6) Feature Selection
Führe alle Selektionsmethoden aus.

In [ ]:
!python ../scripts/create_feature_selection.py

In [ ]:
from FIT_python.config import PROCESSED_DIR, DEFAULT_TARGETS, normalize_dataset_name
from pathlib import Path

print('Feature-Selection Outputs:')
found = False
for ds in PROCESSED_DIR.iterdir():
    fs_base = ds / 'feature_selection'
    if fs_base.exists():
        found = True
        print(f'  {normalize_dataset_name(ds.name)}:')
        for target in DEFAULT_TARGETS:
            tgt_dir = fs_base / target
            if tgt_dir.exists():
                methods = sorted(p.name for p in tgt_dir.glob('*.txt'))
                print(f'    {target}: {methods}')
            else:
                print(f'    {target}: kein Ordner')
assert found, 'No feature selection results'


## 7) Dimensionality Reduction
Berechne PCA, t-SNE und UMAP für jede Datenbasis.

In [ ]:
!python ../scripts/create_dim_reduction.py

In [ ]:
from FIT_python.config import PROCESSED_DIR
files = list(PROCESSED_DIR.rglob('dim_reduced/*.parquet'))
for p in files:
    print(p.relative_to(PROCESSED_DIR))
assert files, 'No dim reduction results'
